## Deep Research

One of the classic cross-business Agentic use cases! This is huge.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">A Deep Research agent is broadly applicable to any business area, and to your own day-to-day activities. You can make use of this yourself!
            </span>
        </td>
    </tr>
</table>

In [ ]:
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict
from IPython.display import display, Markdown
from google import genai
from google.genai import types

In [ ]:
load_dotenv(override=True)
client = genai.Client()
MODEL_ID = 'gemini-2.5-flash'

## Gemini Search Tool

The Gemini API includes a native `google_search` tool that allows models to query Google Search directly and incorporate real-time results into their responses.

In [ ]:
INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 words. Capture the main points. Write succintly, no need to have complete sentences or good grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the essence and ignore any fluff. Do not include any additional commentary other than the summary itself."

async def search_agent_run(message: str) -> str:
    response = await client.aio.models.generate_content(
        model=MODEL_ID,
        contents=message,
        config=types.GenerateContentConfig(
            system_instruction=INSTRUCTIONS,
            tools=[{"google_search": {}}],
            temperature=0.0
        )
    )
    return response.text

In [ ]:
message = "Latest AI Agent frameworks in 2025"

print("Searching...")
result = await search_agent_run(message)
display(Markdown(result))

### We will now use Structured Outputs, and include a description of the fields

In [ ]:
HOW_MANY_SEARCHES = 3

INSTRUCTIONS = f"You are a helpful research assistant. Given a query, come up with a set of web searches to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."

class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")

class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")

async def planner_agent_run(message: str) -> WebSearchPlan:
    response = await client.aio.models.generate_content(
        model=MODEL_ID,
        contents=message,
        config=types.GenerateContentConfig(
            system_instruction=INSTRUCTIONS,
            response_mime_type="application/json",
            response_schema=WebSearchPlan,
            temperature=0.0
        )
    )
    return WebSearchPlan.model_validate_json(response.text)

In [ ]:
message = "Latest AI Agent frameworks in 2025"

print("Planning searches...")
result = await planner_agent_run(message)
print(result)

In [ ]:
def send_email(subject: str, html_body: str) -> str:
    """ Send out an email with the given subject and HTML body """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("ed@edwarddonner.com") # Change this to your verified email
    to_email = To("ed.donner@gmail.com") # Change this to your email
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return "success"


In [ ]:
INSTRUCTIONS = """You are able to send a nicely formatted HTML email based on a detailed report.
You will be provided with a detailed report. You should use your tool to send one email, providing the report converted into clean, well presented HTML with an appropriate subject line."""

async def email_agent_run(report_markdown: str):
    chat = client.aio.chats.create(
        model=MODEL_ID,
        config=types.GenerateContentConfig(
            system_instruction=INSTRUCTIONS,
            tools=[send_email],
            temperature=0.0
        )
    )
    response = await chat.send_message(report_markdown)
    return response.text

In [ ]:
INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
)

class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The final report")
    follow_up_questions: list[str] = Field(description="Suggested topics to research further")

async def writer_agent_run(input_text: str) -> ReportData:
    response = await client.aio.models.generate_content(
        model=MODEL_ID,
        contents=input_text,
        config=types.GenerateContentConfig(
            system_instruction=INSTRUCTIONS,
            response_mime_type="application/json",
            response_schema=ReportData,
            temperature=0.0
        )
    )
    return ReportData.model_validate_json(response.text)

### The next 3 functions will plan and execute the search, using planner_agent and search_agent

In [ ]:
async def plan_searches(query: str):
    """ Use the planner_agent to plan which searches to run for the query """
    print("Planning searches...")
    result = await planner_agent_run(f"Query: {query}")
    print(f"Will perform {len(result.searches)} searches")
    return result

async def perform_searches(search_plan: WebSearchPlan):
    """ Call search() for each item in the search plan """
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results

async def search(item: WebSearchItem):
    """ Use the search agent to run a web search for each item in the search plan """
    input_text = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await search_agent_run(input_text)
    return result

### The next 2 functions write a report and email it

In [ ]:
async def write_report(query: str, search_results: list[str]):
    """ Use the writer agent to write a report based on the search results"""
    print("Thinking about report...")
    input_text = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await writer_agent_run(input_text)
    print("Finished writing report")
    return result

async def execute_send_email(report: ReportData):
    """ Use the email agent to send an email with the report """
    print("Writing email...")
    result = await email_agent_run(report.markdown_report)
    print("Email sent")
    return report

### Showtime!

In [ ]:
query ="Latest AI Agent frameworks in 2025"

print("Starting research...")
search_plan = await plan_searches(query)
search_results = await perform_searches(search_plan)
report = await write_report(query, search_results)
await execute_send_email(report)  
print("Hooray!")